# 4. Statistiques et régression linéaire

On utilise `simple-statistics` pour les statistiques descriptives et `ml-regression` pour la régression linéaire.

## 4.1 Charger les données

In [ ]:
import pl from "nodejs-polars";

// inferSchemaLength : Polars lit 1000 lignes pour deviner le type
// de chaque colonne (sinon "mpg" est pris pour un entier et la
// lecture échoue sur 17.5).
const df = pl.readCSV("../data/auto-mpg.csv", {
  inferSchemaLength: 1000,
});
console.log(df.head(5).toString());
console.log("Dimensions :", df.shape);

## 4.2 Statistiques descriptives avec Polars

In [ ]:
// Résumé statistique des colonnes numériques
console.log(df.describe().toString());

// Moyenne et écart-type d'une colonne
const mpgMean = df.getColumn("mpg").mean();
const mpgStd = df.select(pl.col("mpg").std()).getColumn("mpg").toArray()[0];
console.log(`MPG moyen : ${mpgMean.toFixed(2)}`);
console.log(`Écart-type MPG : ${mpgStd.toFixed(2)}`);

## 4.3 Statistiques avec simple-statistics

In [ ]:
import * as ss from "simple-statistics";

const mpg = df.getColumn("mpg").toArray();
const weight = df.getColumn("weight").toArray();

console.log("Moyenne :", ss.mean(mpg).toFixed(2));
console.log("Médiane :", ss.median(mpg));
// Variance et écart-type d'échantillon (même calcul que Polars)
console.log("Variance :", ss.sampleVariance(mpg).toFixed(2));
console.log("Écart-type :", ss.sampleStandardDeviation(mpg).toFixed(2));
const correlation = ss.sampleCorrelation(weight, mpg);
console.log("Corrélation poids/MPG :", correlation.toFixed(4));

## 4.4 Régression linéaire simple avec simple-statistics

In [ ]:
// Préparer les données sous la forme [[x, y], ...]
const points = weight.map((w, i) => [w, mpg[i]]);

const regression = ss.linearRegression(points);
const slope = regression.m;
const intercept = regression.b;

const a = slope.toFixed(6);
const b = intercept.toFixed(4);
console.log(`Pente (a) : ${a}`);
console.log(`Ordonnée à l'origine (b) : ${b}`);
console.log(`Équation : MPG = ${a} × weight + ${b}`);

// Coefficient de détermination R²
const r2 = ss.rSquared(points, (x) => slope * x + intercept);
console.log(`R² : ${r2.toFixed(4)}`);

// Prédictions
for (const w of [2000, 3000, 4000, 5000]) {
  const prediction = slope * w + intercept;
  console.log(`Poids ${w} lbs → MPG prédit : ${prediction.toFixed(2)}`);
}

## 4.5 Régression linéaire avec ml-regression

In [ ]:
import { SimpleLinearRegression } from "ml-regression";

const regression2 = new SimpleLinearRegression(weight, mpg);

console.log(`Pente : ${regression2.slope.toFixed(6)}`);
console.log(`Ordonnée : ${regression2.intercept.toFixed(4)}`);
const prediction3000 = regression2.predict(3000);
console.log(`Prédiction pour 3000 lbs : ${prediction3000.toFixed(2)}`);

// Score R²
const score = regression2.score(weight, mpg); // { r, r2, chi2, rmsd }
console.log(`Score R² : ${score.r2.toFixed(4)}`);

## 4.6 Visualiser la régression avec Plotly

On reprend la fonction `plotly(data, layout, nom)` du notebook 3, avec ses boutons de téléchargement.

In [ ]:
// data : séries du graphique ; layout : titre, axes...
// nom : nom du fichier téléchargé avec les boutons PNG et SVG.
function plotly(data: object[], layout: object = {}, nom = "graphique") {
  const id = `plot-${crypto.randomUUID()}`;
  const html = `
<div id="${id}" style="width: 100%; max-width: 750px; height: 450px;"></div>
<div style="margin: 4px 0 16px;">
  <button id="${id}-png">⬇ PNG</button>
  <button id="${id}-svg">⬇ SVG</button>
</div>
<script>
  (function () {
    const id = ${JSON.stringify(id)};
    const data = ${JSON.stringify(data)};
    const layout = ${JSON.stringify(layout)};
    const nom = ${JSON.stringify(nom)};

    function draw() {
      Plotly.newPlot(id, data, layout, {
        responsive: true,
        toImageButtonOptions: { filename: nom, scale: 2 },
      });
      for (const format of ["png", "svg"]) {
        const button = document.getElementById(id + "-" + format);
        button.onclick = () =>
          Plotly.downloadImage(id, { format, filename: nom, scale: 2 });
      }
    }

    if (window.Plotly) return draw();
    const script = document.createElement("script");
    script.src = "https://cdn.plot.ly/plotly-2.35.2.min.js";
    script.onload = draw;
    document.head.appendChild(script);
  })();
</script>`;
  Deno.jupyter.display({ "text/html": html }, { raw: true });
}

In [ ]:
const minWeight = Math.min(...weight);
const maxWeight = Math.max(...weight);
const lineX = [minWeight, maxWeight];
const lineY = lineX.map((w) => slope * w + intercept);

const data = [
  {
    type: "scatter",
    mode: "markers",
    name: "Données",
    x: weight,
    y: mpg,
    marker: { color: "#36a2eb", size: 6 },
  },
  {
    type: "scatter",
    mode: "lines",
    name: "Régression",
    x: lineX,
    y: lineY,
    line: { color: "#ff6384", width: 3 },
  },
];

const layout = {
  title: "Régression linéaire : MPG vs Poids",
  xaxis: { title: "Poids (lbs)" },
  yaxis: { title: "Consommation (MPG)" },
};

plotly(data, layout, "regression-mpg-poids");

## 4.7 Calculer le MSE et le RMSE

In [ ]:
// Prédictions pour toutes les données
const predictions = weight.map((w) => slope * w + intercept);

// MSE : moyenne des carrés des écarts entre valeurs réelles et prédites
const mse = ss.mean(mpg.map((y, i) => (y - predictions[i]) ** 2));
const rmse = Math.sqrt(mse);

console.log(`MSE : ${mse.toFixed(4)}`);
console.log(`RMSE : ${rmse.toFixed(4)}`);
console.log(`R² : ${r2.toFixed(4)}`);